# 🚀 Pipeline Python pour le Machine Learning

---

## Introduction

Un **Pipeline** est une séquence d'étapes de traitement enchaînées : chaque étape transforme les données avant de les passer à l'étape suivante.

```
Données brutes → Prétraitement → Transformation → Modèle → Prédictions
```

scikit-learn fournit la classe `Pipeline` qui permet de chaîner des **Transformers** et un **Estimator** final.

In [ ]:
# Installation des dépendances (si nécessaire)
# !pip install scikit-learn numpy pandas matplotlib seaborn joblib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import load_iris, load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

sns.set_theme(style='whitegrid', palette='muted')
print('Bibliothèques chargées ✓')

---
## 1. Pipeline Simple : StandardScaler + LogisticRegression

Le pipeline le plus basique : normaliser les données puis appliquer un classificateur.

In [ ]:
# Chargement du jeu de données Iris
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Création du pipeline
pipeline_simple = Pipeline(steps=[
    ('scaler',     StandardScaler()),
    ('classifier', LogisticRegression(max_iter=200, random_state=42))
])

# Entraînement
pipeline_simple.fit(X_train, y_train)

# Évaluation
score = pipeline_simple.score(X_test, y_test)
cv    = cross_val_score(pipeline_simple, X, y, cv=5)

print(f'Précision test set   : {score:.4f}')
print(f'Validation croisée   : {cv.mean():.4f} ± {cv.std():.4f}')

In [ ]:
# Matrice de confusion
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    pipeline_simple, X_test, y_test,
    display_labels=iris.target_names,
    cmap='Blues', ax=ax
)
ax.set_title('Matrice de confusion — Pipeline Simple')
plt.tight_layout()
plt.show()

---
## 2. Pipeline avec PCA (Réduction de Dimensionnalité)

Enchaînement : `StandardScaler → PCA → SVM`

In [ ]:
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

pipeline_pca = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('pca',    PCA(n_components=10)),
    ('svm',    SVC(kernel='rbf', random_state=42))
])

pipeline_pca.fit(X_train_c, y_train_c)
score_pca = pipeline_pca.score(X_test_c, y_test_c)
var_exp   = pipeline_pca['pca'].explained_variance_ratio_.sum()

print(f'Features originales      : {X_c.shape[1]}')
print(f'Composantes PCA retenues : 10')
print(f'Variance expliquée       : {var_exp:.4f} ({var_exp*100:.2f}%)')
print(f'Précision                : {score_pca:.4f}')

In [ ]:
# Visualisation de la variance expliquée
pipe_var = Pipeline([
    ('scaler', StandardScaler()),
    ('pca',    PCA())
])
pipe_var.fit(X_c)
cumvar = np.cumsum(pipe_var['pca'].explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(cumvar)+1), cumvar, marker='o', linewidth=2)
ax.axhline(0.95, color='red', linestyle='--', label='95% variance')
ax.axvline(10,   color='green', linestyle='--', label='10 composantes')
ax.set_xlabel('Nombre de composantes')
ax.set_ylabel('Variance cumulée expliquée')
ax.set_title('PCA — Variance cumulée expliquée')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Pipeline avec Données Mixtes (ColumnTransformer)

Traitement différencié pour colonnes numériques et catégorielles.

In [ ]:
np.random.seed(42)
n = 300
df = pd.DataFrame({
    'age':         np.random.randint(18, 70, n).astype(float),
    'salaire':     np.random.normal(45000, 15000, n),
    'experience':  np.random.randint(0, 40, n).astype(float),
    'niveau':      np.random.choice(['Débutant', 'Intermédiaire', 'Expert'], n),
    'département': np.random.choice(['Ventes', 'Tech', 'RH', 'Finance'], n),
})
# Valeurs manquantes artificielles
df.loc[df.sample(frac=0.05, random_state=1).index, 'age']    = np.nan
df.loc[df.sample(frac=0.05, random_state=2).index, 'salaire'] = np.nan

y_mix = (df['salaire'].fillna(df['salaire'].median()) > 45000).astype(int)

print('Aperçu du dataset :')
display(df.head())
print(f'\nValeurs manquantes :\n{df.isnull().sum()}')

In [ ]:
numeric_features     = ['age', 'salaire', 'experience']
categorical_features = ['niveau', 'département']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,     numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

pipeline_mix = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42))
])

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    df, y_mix, test_size=0.2, random_state=42
)
pipeline_mix.fit(X_train_m, y_train_m)
print(f'Précision : {pipeline_mix.score(X_test_m, y_test_m):.4f}')

---
## 4. Optimisation des Hyperparamètres avec GridSearchCV

GridSearchCV peut directement optimiser les paramètres de chaque étape du pipeline.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline_gs = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('svc',    SVC(random_state=42))
])

# Notation avec __ pour accéder aux paramètres d'une étape
param_grid = {
    'svc__C':      [0.1, 1, 10, 100],
    'svc__kernel': ['linear', 'rbf'],
    'svc__gamma':  ['scale', 'auto'],
}

grid_search = GridSearchCV(pipeline_gs, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f'Meilleurs paramètres : {grid_search.best_params_}')
print(f'Meilleur score CV    : {grid_search.best_score_:.4f}')
print(f'Score test set       : {grid_search.score(X_test, y_test):.4f}')

---
## 5. Comparaison de Plusieurs Pipelines

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipelines = {
    'Logistic Regression':  Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=200))]),
    'KNN':                  Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier())]),
    'SVM':                  Pipeline([('sc', StandardScaler()), ('clf', SVC(random_state=42))]),
    'Random Forest':        Pipeline([('sc', MinMaxScaler()),   ('clf', RandomForestClassifier(random_state=42))]),
    'Gradient Boosting':    Pipeline([('sc', StandardScaler()), ('clf', GradientBoostingClassifier(random_state=42))]),
}

results = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    cv = cross_val_score(pipe, X, y, cv=5)
    results[name] = {'test': pipe.score(X_test, y_test), 'cv_mean': cv.mean(), 'cv_std': cv.std()}

results_df = pd.DataFrame(results).T
print(results_df.sort_values('cv_mean', ascending=False).to_string())

In [ ]:
# Visualisation de la comparaison
fig, ax = plt.subplots(figsize=(9, 5))
names  = list(results.keys())
means  = [results[n]['cv_mean'] for n in names]
stds   = [results[n]['cv_std']  for n in names]
colors = sns.color_palette('muted', len(names))

bars = ax.barh(names, means, xerr=stds, color=colors, capsize=5, edgecolor='grey')
ax.set_xlabel('Précision (Validation croisée 5-fold)')
ax.set_title('Comparaison des Pipelines ML — Iris Dataset')
ax.set_xlim(0.8, 1.02)
for bar, mean in zip(bars, means):
    ax.text(mean + 0.001, bar.get_y() + bar.get_height()/2,
            f'{mean:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 6. Sauvegarde et Chargement d'un Pipeline

In [ ]:
# Sauvegarde
joblib.dump(pipeline_simple, '/tmp/pipeline_demo.joblib')
print('Pipeline sauvegardé ✓')

# Chargement
pipeline_loaded = joblib.load('/tmp/pipeline_demo.joblib')
print(f'Pipeline rechargé  ✓')
print(f'Étapes : {[s[0] for s in pipeline_loaded.steps]}')
print(f'Précision (après rechargement) : {pipeline_loaded.score(X_test, y_test):.4f}')

---
## Résumé

| Concept | Rôle |
|---|---|
| `Pipeline` | Chaîne ordonnée d'étapes de traitement et d'un modèle |
| `StandardScaler` | Normalisation (moyenne=0, écart-type=1) |
| `PCA` | Réduction de dimensionnalité |
| `SimpleImputer` | Gestion des valeurs manquantes |
| `OneHotEncoder` | Encodage des variables catégorielles |
| `ColumnTransformer` | Traitement différencié par type de colonne |
| `SelectKBest` | Sélection des k meilleures features |
| `GridSearchCV` | Optimisation des hyperparamètres |
| `joblib` | Sauvegarde et chargement du pipeline |

### Bonnes pratiques
- Toujours ajuster (`fit`) le pipeline sur les données d'**entraînement** uniquement
- Utiliser `cross_val_score` ou `GridSearchCV` directement sur le pipeline
- Sauvegarder le pipeline complet (pas seulement le modèle) pour la mise en production